# Preprocesamiento de imágenes — ISIC 2020

**CNNs vs Transformers para clasificación de melanoma**

Este notebook se ejecuta **una sola vez**. Su objetivo es convertir las 33,126 imágenes dermatoscópicas originales de ISIC 2020 (resolución completa, hasta 6000×4000 px) en una versión reducida que podamos cargar rápido durante el entrenamiento.

Pasos:
1. Cargar las etiquetas (`GroundTruth_v2.csv`) y la lista de duplicados.
2. Eliminar las imágenes duplicadas.
3. Redimensionar cada imagen (lado corto = 256 px, conservando la proporción) leyendo directamente del zip, sin descomprimirlo.
4. Guardar un CSV con las etiquetas y metadatos de las imágenes finales.
5. Verificar visualmente el resultado.

> **¿Por qué lado corto = 256 y no un cuadrado 256×256?** Las imágenes tienen proporciones distintas (3:2, 16:9, 4:3). Si recortamos ahora, perdemos información de forma permanente. Al conservar la proporción, el recorte a 224×224 se decide después en los `transforms` de entrenamiento (`RandomResizedCrop` / `CenterCrop`).

In [1]:
## Standard libraries
import os
import io
import time
import zipfile
import threading
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

## Imágenes
from PIL import Image

## Imports for plotting
import matplotlib.pyplot as plt
%matplotlib inline
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg', 'pdf') # For export
import seaborn as sns
sns.set()

## Progress bar
from tqdm.notebook import tqdm

## Configuración

Las imagenes descargadas se alamcenan en el folder (`C:\datasets\isic2020`), junto con la informacion de duplicados e informacion de las imagenes respectivas, al final creamos el folder `im256` en el que almancenaremos las imagenes redimensionadas.

In [2]:
# Carpeta con el zip y los CSV descargados de https://challenge.isic-archive.com/data/#2020
DATASET_PATH = r"C:\datasets\isic2020"
ZIP_PATH     = os.path.join(DATASET_PATH, "ISIC_2020_Training_JPEG.zip")
GT_PATH      = os.path.join(DATASET_PATH, "ISIC_2020_Training_GroundTruth_v2.csv")
DUP_PATH     = os.path.join(DATASET_PATH, "ISIC_2020_Training_Duplicates.csv")

# Salida: imágenes redimensionadas + CSV de etiquetas
IMG_SIZE     = 256                                   # lado corto en píxeles
OUT_DIR      = os.path.join(DATASET_PATH, f"img{IMG_SIZE}")
OUT_CSV      = os.path.join(DATASET_PATH, f"train_{IMG_SIZE}.csv")
JPEG_QUALITY = 95
NUM_WORKERS  = 8                                     # hilos para el procesamiento en paralelo

os.makedirs(OUT_DIR, exist_ok=True)
for p in [ZIP_PATH, GT_PATH, DUP_PATH]:
    assert os.path.isfile(p), f"No se encontró {p}"
print("Salida:", OUT_DIR)

Salida: C:\datasets\isic2020\img256


## 1. Etiquetas y metadatos

Urilizando Pandas, leemos el archivo `GroundTruth_v2.csv` que contiene por imagen: paciente, lesión, sexo, edad aproximada, sitio anatómico, diagnóstico y la etiqueta binaria `target` (1 = melanoma).

In [3]:
gt = pd.read_csv(GT_PATH)
print(f"Imágenes: {len(gt):,}  |  Pacientes: {gt.patient_id.nunique():,}  |  Lesiones: {gt.lesion_id.nunique():,}")
print(f"Malignas: {gt.target.sum():,} ({gt.target.mean():.2%})")
gt.head()

Imágenes: 33,126  |  Pacientes: 2,056  |  Lesiones: 32,701
Malignas: 584 (1.76%)


,image_name,patient_id,lesion_id,sex,age_approx,anatom_site_general_challenge,diagnosis,benign_malignant,target
0,ISIC_2637011,IP_7279968,IL_7972535,male,45.0,head/neck,unknown,benign,0
1,ISIC_0015719,IP_3075186,IL_4649854,female,45.0,upper extremity,unknown,benign,0
2,ISIC_0052212,IP_2842074,IL_9087444,female,50.0,lower extremity,nevus,benign,0
3,ISIC_0068279,IP_6890425,IL_4255399,female,45.0,head/neck,unknown,benign,0
4,ISIC_0074268,IP_8723313,IL_6898037,female,55.0,upper extremity,unknown,benign,0


## 2. Eliminación de duplicados

La lista de 425 pares de imágenes duplicadas. Si dejamos ambas copias, una podría caer en entrenamiento y la otra en prueba, inflando las métricas por lo cual procedemos a confirmar antes de eliminar y asi verificamos que los pares sean consistentes.

In [4]:
dup = pd.read_csv(DUP_PATH)
gt_idx = gt.set_index("image_name")

t1 = gt_idx.loc[dup.image_name_1, "target"].values
t2 = gt_idx.loc[dup.image_name_2, "target"].values
same_patient = gt_idx.loc[dup.image_name_1, "patient_id"].values == gt_idx.loc[dup.image_name_2, "patient_id"].values
in_many = (pd.concat([dup.image_name_1, dup.image_name_2]).value_counts() > 1).sum()

print(f"Pares duplicados:            {len(dup)}")
print(f"Pares con etiqueta distinta: {(t1 != t2).sum()}")
print(f"Pares del mismo paciente:    {same_patient.mean():.0%}")
print(f"Imágenes en más de un par:   {in_many}")

Pares duplicados:            425
Pares con etiqueta distinta: 0
Pares del mismo paciente:    100%
Imágenes en más de un par:   0


Ningún par tiene etiquetas contradictorias y ninguna imagen aparece en más de un par, así que basta con conservar `image_name_1` y eliminar `image_name_2`.

In [5]:
df = gt[~gt.image_name.isin(dup.image_name_2)].reset_index(drop=True)
print(f"Imágenes después de quitar duplicados: {len(df):,}  (se eliminaron {len(gt) - len(df)})")
print(f"Malignas: {df.target.sum():,} ({df.target.mean():.2%})")

Imágenes después de quitar duplicados: 32,701  (se eliminaron 425)
Malignas: 581 (1.78%)


## 3. Redimensionamiento

Para cada imagen:
1. Se lee directamente del zip asi evitamos descomprimir el archivo de 42GB
2. `Image.draft()` le pide al decodificador JPEG que decodifique ya reducida (escalas 1/2, 1/4, 1/8). Esto acelera muchísimo las imágenes de 6000×4000.
3. Se redimensiona con filtro Lanczos para que el lado corto mida `IMG_SIZE` 256.
4. Se guarda como JPEG de calidad 95.

In [ ]:
_local = threading.local()

def get_zip():
    # Un ZipFile por hilo
    if not hasattr(_local, "zf"):
        _local.zf = zipfile.ZipFile(ZIP_PATH)
    return _local.zf

def resize_shorter_side(img, size):
    w, h = img.size
    scale = size / min(w, h)
    new_w, new_h = round(w * scale), round(h * scale)
    return img.resize((new_w, new_h), Image.LANCZOS)

def process_image(image_name):
    out_path = os.path.join(OUT_DIR, f"{image_name}.jpg")
    try:
        with get_zip().open(f"train/{image_name}.jpg") as f:
            img = Image.open(io.BytesIO(f.read()))
            orig_w, orig_h = img.size
            if os.path.exists(out_path):                 # ya procesada: solo leemos el tamaño original
                return image_name, orig_w, orig_h, None
            img.draft("RGB", (IMG_SIZE, IMG_SIZE))       
            img = resize_shorter_side(img.convert("RGB"), IMG_SIZE)
            img.save(out_path, "JPEG", quality=JPEG_QUALITY)
        return image_name, orig_w, orig_h, None
    except Exception as e:
        return image_name, None, None, repr(e)

def run(names, num_workers=NUM_WORKERS):
    with ThreadPoolExecutor(max_workers=num_workers) as ex:
        results = list(tqdm(ex.map(process_image, names), total=len(names)))
    return pd.DataFrame(results, columns=["image_name", "orig_width", "orig_height", "error"])

### 3.2 Procesar todas las imágenes

Este paso tarda varios minutos (depende del disco y del CPU).

In [9]:
start = time.time()
res = run(df.image_name.tolist())
elapsed = time.time() - start
print(f"Tiempo total: {elapsed / 60:.1f} min  ({len(res) / elapsed:.0f} imágenes/s)")
print(f"Errores: {res.error.notna().sum()}")
res[res.error.notna()]

  0%|          | 0/32701 [00:00<?, ?it/s]

Tiempo total: 2.2 min  (248 imágenes/s)
Errores: 0


,image_name,orig_width,orig_height,error


## 4. Verificación y CSV final

Comprobamos que exista una imagen por cada fila y guardamos el CSV que usarán los siguientes notebooks.

In [10]:
assert res.error.isna().all(), "Hay imágenes con error; revisar la tabla anterior"

files = {f[:-4] for f in os.listdir(OUT_DIR) if f.endswith(".jpg")}
missing = set(df.image_name) - files
print(f"Imágenes esperadas: {len(df):,}  |  En disco: {len(files):,}  |  Faltantes: {len(missing)}")

size_gb = sum(os.path.getsize(os.path.join(OUT_DIR, f)) for f in os.listdir(OUT_DIR)) / 1024**3
print(f"Tamaño de {OUT_DIR}: {size_gb:.2f} GB")

Imágenes esperadas: 32,701  |  En disco: 32,701  |  Faltantes: 0
Tamaño de C:\datasets\isic2020\img256: 0.86 GB


In [11]:
final = df.merge(res[["image_name", "orig_width", "orig_height"]], on="image_name")
final["image_path"] = final.image_name.apply(lambda n: os.path.join(OUT_DIR, f"{n}.jpg"))
final.to_csv(OUT_CSV, index=False)
print("Guardado:", OUT_CSV, final.shape)
final.head()

Guardado: C:\datasets\isic2020\train_256.csv (32701, 12)


,image_name,patient_id,lesion_id,sex,age_approx,anatom_site_general_challenge,diagnosis,benign_malignant,target,orig_width,orig_height,image_path
0,ISIC_2637011,IP_7279968,IL_7972535,male,45.0,head/neck,unknown,benign,0,6000,4000,C:\datasets\isic2020\img256\ISIC_2637011.jpg
1,ISIC_0015719,IP_3075186,IL_4649854,female,45.0,upper extremity,unknown,benign,0,6000,4000,C:\datasets\isic2020\img256\ISIC_0015719.jpg
2,ISIC_0052212,IP_2842074,IL_9087444,female,50.0,lower extremity,nevus,benign,0,1872,1053,C:\datasets\isic2020\img256\ISIC_0052212.jpg
3,ISIC_0068279,IP_6890425,IL_4255399,female,45.0,head/neck,unknown,benign,0,1872,1053,C:\datasets\isic2020\img256\ISIC_0068279.jpg
4,ISIC_0074268,IP_8723313,IL_6898037,female,55.0,upper extremity,unknown,benign,0,6000,4000,C:\datasets\isic2020\img256\ISIC_0074268.jpg
